# ML Challenge: Customer Outcome Prediction (Luca)

## Ziel
Entwickeln Sie ein Klassifikationsmodell zur Vorhersage von `Target`.

### Aufgaben
1. Daten explorieren
2. Missing Values behandeln
3. Kategoriale Variablen kodieren
4. Mindestens 3 Modelle trainieren (nutzen auch Cross Validation) und base version erstellen
5. Accuracy, Precision, Recall und F1 vergleichen
6. Hyperparameter Tuning machen und GridCV
7. Accuracy, Precision, Recall und F1 vergleichen
8. Ergebnisse interpretieren


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


In [10]:
df = pd.read_csv('kunden.csv')  # Reads csv file into a dataframe
df.head()  # Outputs the first 5 rows of csv file


,PersonID,Target,ServiceLevel,FullName,Gender,Age,RelativesOnboard,Dependents,BookingID,PricePaid,Room,DepartureLocation
0,1,0,Basic,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,A
1,2,1,Premium,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,B
2,3,1,Basic,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,A
3,4,1,Premium,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,A
4,5,0,Basic,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,A


## 1. Exploration

In [12]:
print('Shape of Data: ')
print(df.shape)

print('\nAll columns and their datatypes in csv file: ')
print(df.dtypes)

print('\nAll columns that have missing values: ')
print(df.isna().sum())

print('\nStatistical conclusion of all data : ')
print('\n- shows things like count of all columns that have values : ')
df.describe(include='all')

Shape of Data: 
(891, 12)

All datatypes of data in csv file: 
PersonID               int64
Target                 int64
ServiceLevel             str
FullName                 str
Gender                   str
Age                  float64
RelativesOnboard       int64
Dependents             int64
BookingID                str
PricePaid            float64
Room                     str
DepartureLocation        str
dtype: object

All columns that have missing values: 
PersonID               0
Target                 0
ServiceLevel           0
FullName               0
Gender                 0
Age                  177
RelativesOnboard       0
Dependents             0
BookingID              0
PricePaid              0
Room                 687
DepartureLocation      2
dtype: int64

=== Statistische Zusammenfassung ===


,PersonID,Target,ServiceLevel,FullName,Gender,Age,RelativesOnboard,Dependents,BookingID,PricePaid,Room,DepartureLocation
count,891.000000,891.000000,891,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,3,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,Basic,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,A
freq,NaN,NaN,491,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,NaN,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,NaN,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,NaN,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,NaN,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,NaN,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,NaN,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


In [35]:
print('- Target Distribution: ')
print(df['Target'].value_counts())

print('\n- Target Distribution (%): ')
print(f"Target=0: {df['Target'].value_counts()[0]} ({df['Target'].value_counts(normalize=True)[0] * 100:.1f}%)")
print(f"Target=1: {df['Target'].value_counts()[1]} ({df['Target'].value_counts(normalize=True)[1] * 100:.1f}%)")

print("------------------------")

print('\n- ServiceLevel Distribution: ')
print(df['ServiceLevel'].value_counts())

print('\n- ServiceLevel Distribution (%): ')
for service_level, count in df['ServiceLevel'].value_counts().items():
    print(f"ServiceLevel={service_level}: {count} "f"({df['ServiceLevel'].value_counts(normalize=True)[service_level] * 100:.1f}%)")

print("------------------------")

print('\n- Gender Distribution: ')
print(df['Gender'].value_counts())

print('\n- Gender Distribution (%): ')
for gender, count in df['Gender'].value_counts().items():
    print(f"Gender={gender}: {count} "f"({df['Gender'].value_counts(normalize=True)[gender] *100:.1f}%)")

print("------------------------")

print('\n- DepartureLocation Distribution: ')
print(df['DepartureLocation'].value_counts())

print('\n- DepartureLocation Distribution (%): ')
for location, count in df['DepartureLocation'].value_counts().items():
    percentage = df['DepartureLocation'].value_counts(normalize=True)[location] * 100
    print(f"DepartureLocation={location}: {count} ({percentage:.1f}%)")


- Target Distribution: 
Target
0    549
1    342
Name: count, dtype: int64

- Target Distribution (%): 
Target=0: 549 (61.6%)
Target=1: 342 (38.4%)
------------------------

- ServiceLevel Distribution: 
ServiceLevel
Basic       491
Premium     216
Standard    184
Name: count, dtype: int64

- ServiceLevel Distribution (%): 
ServiceLevel=Basic: 491 (55.1%)
ServiceLevel=Premium: 216 (24.2%)
ServiceLevel=Standard: 184 (20.7%)
------------------------

- Gender Distribution: 
Gender
male      577
female    314
Name: count, dtype: int64

- Gender Distribution (%): 
Gender=male: 577 (64.8%)
Gender=female: 314 (35.2%)
------------------------

- DepartureLocation Distribution: 
DepartureLocation
A    644
B    168
C     77
Name: count, dtype: int64

- DepartureLocation Distribution (%): 
DepartureLocation=A: 644 (72.4%)
DepartureLocation=B: 168 (18.9%)
DepartureLocation=C: 77 (8.7%)


## 2. Features und Target definieren

In [39]:
# Delete columns that have no value as a feature for the prediction:
# - PersonID: primary key -> no predictor
# - FullName: title can be extracted (Mr., Miss. ,etc..) -> rest of name is no predictor
# - BookingID: unique key -> no predictor
# - Room: too many missing values for it being a predictor

# Extract title from fullname with regex
df['Title'] = df['FullName'].str.extract(r',\s([A-Za-z]+)\.', expand=False)
print('Extracted titles:')
print(df['Title'].value_counts())

# Give rare title (<9 : 10% of all data rows) to "other"
title_counts = df['Title'].value_counts()
rare_titles = title_counts[title_counts < 9].index

def replace_rare_titles(x):
    if x in rare_titles:
        return 'Other'
    else:
        return x


df['Title'] = df['Title'].apply(replace_rare_titles)
print('\nCleaned titles:')
print(df['Title'].value_counts())

print('\nCleaned titles (%):')
for title, count in df['Title'].value_counts().items():
    print(f"Title={title}: {count} "f"({df['Title'].value_counts(normalize=True)[title] *100:.1f}%)")




Extracted titles:
Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Jonkheer      1
Name: count, dtype: int64

Cleaned titles:
Title
Mr        517
Miss      182
Mrs       125
Master     40
Other      26
Name: count, dtype: int64

Cleaned titles (%):
Title=Mr: 517 (58.1%)
Title=Miss: 182 (20.4%)
Title=Mrs: 125 (14.0%)
Title=Master: 40 (4.5%)
Title=Other: 26 (2.9%)


In [40]:
# Define feature set
drop_cols = ['PersonID', 'FullName', 'BookingID', 'Room']
df_model = df.drop(columns=drop_cols)

# Split the data into the input data for later for the model (x, data without solution "target")
# and the output data with which the model should learn and get better (y, just the "target" column)
X = df_model.drop(columns=['Target'])
y = df_model['Target']

print('Features:', X.columns.tolist())
print('Shape X:', X.shape)
print('Shape y:', y.shape)

Features: ['ServiceLevel', 'Gender', 'Age', 'RelativesOnboard', 'Dependents', 'PricePaid', 'DepartureLocation', 'Title']
Shape X: (891, 8)
Shape y: (891,)


## 3. Preprocessing definieren

In [41]:
# Split features up into different kinds

# Numeric features: columns with numeric values, where distances are useful
numeric_features = ['Age', 'RelativesOnboard', 'Dependents', 'PricePaid']

# Ordinal features: column that has values with natural order but the distances can not be exactly
# measured
# ServiceLevel: Premium > Standard > Basic
ordinal_features = ['ServiceLevel']
ordinal_categories = [['Basic', 'Standard', 'Premium']]

# Categorical features: features without proper order (male cannot be > female for example)
categorical_features = ['Gender', 'DepartureLocation', 'Title']

print('Numeric features:', numeric_features)
print('Ordinal features:', ordinal_features)
print('Categorical features:', categorical_features)


Numeric features: ['Age', 'RelativesOnboard', 'Dependents', 'PricePaid']
Ordinal features: ['ServiceLevel']
Categorical features: ['Gender', 'DepartureLocation', 'Title']


In [42]:
from sklearn.preprocessing import OrdinalEncoder

# Preprocessing Pipelines

# Numeric transformer: fills up missing values (NaN) with median value of column
# Standard scaler brings up all numeric values from all numeric columns to same scale, so
# the model cant interpret some numeric values from another column as more "important"
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Ordinal transformer: fills up missing values (NaN) with most common value
# OrdinalEncoder transforms the ordinal values into numeric values, for example:
# Basic    ->  0
# Standard ->  1
# Premium  ->  2
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=ordinal_categories))
])

# Categorical transformer: fills up missing values (NaN) with most common value
# OneHotEncoder: creates per catgeory own 0/1 column matrix, for example:
# Gender:
# male    ->  1  0
# female  ->  0  1

# DepartureLocation:
# A  ->  1  0  0
# B  ->  0  1  0
# C  ->  0  0  1

# if there is an unkwnon value in the predicition process, it will be coded as a complete 0 matrix
# => (handle_unknown='ignore')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('Preprocessor ready')

Preprocessor ready


In [43]:
# Train/Test-Split (80/20)
# Use 80% for traing data and 20% for test data
# Use random_state=42, so the distribution is random
# Use stratify on target (y), so the distribution on the target value is on every set the same
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training data: {X_train.shape[0]} Samples')
print(f'Test data:      {X_test.shape[0]} Samples')

Training data: 712 Samples
Test data:      179 Samples


## 4. Modelle trainieren (und verbessern) [Base Versions + Cross Validation]

In [46]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Own function that is called up for every model
# name = Name of model
# model = The sklearn-Modell (Pipeline)
# X_tr and y_tr = training data
# X_te and y_te = test data
# cv=5 = Number of folds for cross validation (standard is 5)
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, cv=5):

    # Model is being trained on test data
    model.fit(X_tr, y_tr)

    # Model gives its prediction on test data (output are the predicted targets)
    y_pred = model.predict(X_te)

    # Test metrics
    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec  = recall_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred)

    # Cross Validation on training data
    # One single train/test split can be misleading, so it is plit into five seperate chunks
    # For example:
    # X_train split into 5 folds:
    # ┌──────┬──────┬──────┬──────┬──────┐
    # │  F1  │  F2  │  F3  │  F4  │  F5  │
    # └──────┴──────┴──────┴──────┴──────┘

    # Run 1: [F1 Test] [F2+F3+F4+F5 Train] -> Accuracy 0.82
    # Run 2: [F2 Test] [F1+F3+F4+F5 Train] -> Accuracy 0.79
    # Run 3: [F3 Test] [F1+F2+F4+F5 Train] -> Accuracy 0.81
    # Run 4: [F4 Test] [F1+F2+F3+F5 Train] -> Accuracy 0.80
    # Run 5: [F5 Test] [F1+F2+F3+F4 Train] -> Accuracy 0.83
    #                                          ─────────────
    #                                       Average: 0.81

    # StratifiedKFold => same as stratify, so the distribution of target values stays the same
    # shuffle=True => shuffles data before folding
    # scoring='accuracy' => accuracy is the metric that is being calculated on every fold
    cv_scores = cross_val_score(model, X_tr, y_tr,
                                cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
                                scoring='accuracy')

    print(f'\n- {name}')
    print(f'  CV Accuracy:  {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
    print(f'  Test Accuracy: {acc:.4f}')
    print(f'  Precision:     {prec:.4f}')
    print(f'  Recall:        {rec:.4f}')
    print(f'  F1-Score:      {f1:.4f}')

    return {
        'Model': name,
        'CV_Accuracy': cv_scores.mean(),
        'CV_Std': cv_scores.std(),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    }


In [47]:
# Base models
results_base = []

# 1. Logistic Regression
# max_iter=1000 => how many optimization steps are allowed max.
lr_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])
results_base.append(evaluate_model('Logistic Regression (Base)', lr_pipe, X_train, y_train, X_test, y_test))

# 2. Decision Tree
dt_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])
results_base.append(evaluate_model('Decision Tree (Base)', dt_pipe, X_train, y_train, X_test, y_test))

# 3. Random Forest
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])
results_base.append(evaluate_model('Random Forest (Base)', rf_pipe, X_train, y_train, X_test, y_test))

# 4. SVM
svm_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(random_state=42))
])
results_base.append(evaluate_model('SVM (Base)', svm_pipe, X_train, y_train, X_test, y_test))


- Logistic Regression (Base)
  CV Accuracy:  0.8202 (+/- 0.0210)
  Test Accuracy: 0.8380
  Precision:     0.8226
  Recall:        0.7391
  F1-Score:      0.7786

- Decision Tree (Base)
  CV Accuracy:  0.7809 (+/- 0.0176)
  Test Accuracy: 0.8324
  Precision:     0.8000
  Recall:        0.7536
  F1-Score:      0.7761

- Random Forest (Base)
  CV Accuracy:  0.8048 (+/- 0.0103)
  Test Accuracy: 0.8156
  Precision:     0.7903
  Recall:        0.7101
  F1-Score:      0.7481

- SVM (Base)
  CV Accuracy:  0.8216 (+/- 0.0224)
  Test Accuracy: 0.8380
  Precision:     0.8125
  Recall:        0.7536
  F1-Score:      0.7820


In [48]:
df_base = pd.DataFrame(results_base)
df_base = df_base.set_index('Model')
df_base.round(4)

,CV_Accuracy,CV_Std,Accuracy,Precision,Recall,F1
Model,,,,,,
Logistic Regression (Base),0.8202,0.0210,0.8380,0.8226,0.7391,0.7786
Decision Tree (Base),0.7809,0.0176,0.8324,0.8000,0.7536,0.7761
Random Forest (Base),0.8048,0.0103,0.8156,0.7903,0.7101,0.7481
SVM (Base),0.8216,0.0224,0.8380,0.8125,0.7536,0.7820


## 5. Hyperparameter Tuning mit GridSearchCV

In [49]:
from sklearn.model_selection import GridSearchCV

# Reusable CV strategy: 5-fold stratified, shuffled for reproducibility
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression Tuning

# Define hyperparameter grid:
# - C: regularization strength (smaller = more regularization = simpler model)
# - solver: optimization algorithm used to fit the model
param_grid_lr = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__solver': ['liblinear', 'lbfgs']
}

lr_pipe_tune = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# GridSearchCV tries all parameter combinations with cross validation
# scoring='f1' because classes are slightly imbalanced (62% / 38%)
# n_jobs=-1 uses all available CPU cores in parallel
grid_lr = GridSearchCV(lr_pipe_tune, param_grid_lr, cv=cv_strategy, scoring='f1', n_jobs=-1)

# Run the search, fits 10 combinations x 5 folds = 50 training runs
grid_lr.fit(X_train, y_train)

# Best parameter combination found by GridSearchCV
print('Best Params LR:', grid_lr.best_params_)

# Best mean F1 score across all folds for the winning combination
print('Best CV F1 LR: ', round(grid_lr.best_score_, 4))


Best Params LR: {'classifier__C': 1, 'classifier__solver': 'liblinear'}
Best CV F1 LR:  0.7605


In [50]:
# Decision Tree Tuning

# Define hyperparameter grid:
# - max_depth: maximum depth of the tree (None = unlimited, risks overfitting)
# - min_samples_split: minimum samples required to split an internal node
# - min_samples_leaf: minimum samples required to be at a leaf node
# all three parameters control tree complexity and prevent overfitting
param_grid_dt = {
    'classifier__max_depth': [3, 5, 7, 10, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

dt_pipe_tune = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# GridSearchCV tries all 45 combinations (5 x 3 x 3) x 5 folds = 225 training runs
# scoring='f1' to handle slight class imbalance
grid_dt = GridSearchCV(dt_pipe_tune, param_grid_dt, cv=cv_strategy, scoring='f1', n_jobs=-1)

# Run the search on training data only
grid_dt.fit(X_train, y_train)

# Best parameter combination found, expect max_depth to be limited (not None)
# since unlimited depth typically overfits
print('Best Params DT:', grid_dt.best_params_)

# Best mean F1 score across all folds for the winning combination
print('Best CV F1 DT: ', round(grid_dt.best_score_, 4))

Best Params DT: {'classifier__max_depth': 5, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2}
Best CV F1 DT:  0.7312


In [51]:
# Random Forest Tuning

# Define hyperparameter grid:
# - n_estimators: number of trees in the forest (more = more stable, but slower)
# - max_depth: maximum depth per tree (None = unlimited, less critical than
#              single Decision Tree because many trees vote together)
# - min_samples_split: minimum samples required to split a node
# - max_features: number of features considered at each split, core idea of
#                 Random Forest, keeps trees diverse and reduces correlation
param_grid_rf = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_split': [2, 5],
    'classifier__max_features': ['sqrt', 'log2']
}

rf_pipe_tune = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# GridSearchCV tries all 36 combinations (3 x 3 x 2 x 2) x 5 folds = 180 training runs
# n_jobs=-1 is especially important here since Random Forest is computationally expensive
grid_rf = GridSearchCV(rf_pipe_tune, param_grid_rf, cv=cv_strategy, scoring='f1', n_jobs=-1)

# Run the search on training data only
grid_rf.fit(X_train, y_train)

# Best parameter combination, n_estimators will likely be higher (more trees = better)
# max_depth likely limited to avoid overfitting individual trees
print('Best Params RF:', grid_rf.best_params_)

# Best mean F1 score across all folds, expect this to be the highest of all models
print('Best CV F1 RF: ', round(grid_rf.best_score_, 4))

Best Params RF: {'classifier__max_depth': 5, 'classifier__max_features': 'sqrt', 'classifier__min_samples_split': 5, 'classifier__n_estimators': 300}
Best CV F1 RF:  0.7665


In [52]:
# SVM Tuning

# Define hyperparameter grid:
# - C: regularization strength (small = softer margin = more generalization)
# - kernel: defines the shape of the decision boundary
#           'linear' = straight boundary, 'rbf' = curved boundary
# - gamma: controls the influence radius of each data point (rbf kernel only)
#          'scale' adapts to data variance, 'auto' uses 1/n_features
param_grid_svm = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['rbf', 'linear'],
    'classifier__gamma': ['scale', 'auto']
}

svm_pipe_tune = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(random_state=42))
])

# GridSearchCV tries all 12 combinations (3 x 2 x 2) x 5 folds = 60 training runs
# SVM is slower per run than tree-based models, so fewer combinations are tested
grid_svm = GridSearchCV(svm_pipe_tune, param_grid_svm, cv=cv_strategy, scoring='f1', n_jobs=-1)

# Run the search on training data only
grid_svm.fit(X_train, y_train)

# Best combination, 'rbf' kernel typically wins over 'linear' on real-world data
print('Best Params SVM:', grid_svm.best_params_)

# Best mean F1 score across all folds for the winning combination
print('Best CV F1 SVM: ', round(grid_svm.best_score_, 4))

Best Params SVM: {'classifier__C': 1, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
Best CV F1 SVM:  0.7645
